## Chapter 2: Text Preprocessing and Embeddings

In this notebook I explore the key concepts from Chapter 2 of 
"Build a Large Language Model From Scratch" by Sebastian Raschka.
The chapter covers how raw text is transformed into numerical 
representations that language models can understand.


In [3]:
import re
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Longitud del texto:", len(raw_text))
print(raw_text[:200])

Longitud del texto: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


In [4]:
preprocessed = re.split(r'([,.?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print("Número total de tokens:", len(preprocessed))
print(preprocessed[:20])

Número total de tokens: 4649
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


## Step 1: Loading and Tokenizing the Text

The text from "The Verdict" by Edith Wharton contains **20,479 characters** 
and after basic tokenization produces **4,649 tokens**.

This step is fundamental because LLMs cannot process raw text directly. 
Every word, punctuation mark and symbol must first be converted into a 
discrete token that can be represented as a number. Without this step, 
there is no way to feed language into a neural network.

In [5]:
all_tokens = sorted(list(set(preprocessed)))
vocab = {token: integer for integer, token in enumerate(all_tokens)}

print("Tamaño del vocabulario:", len(vocab))

Tamaño del vocabulario: 1159


In [6]:
special_tokens = ["<|unk|>", "<|endoftext|>"]

for tok in special_tokens:
    if tok not in vocab:
        vocab[tok] = len(vocab)

print("Nuevo tamaño del vocabulario:", len(vocab))

Nuevo tamaño del vocabulario: 1161


In [7]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [
            self.str_to_int[s] if s in self.str_to_int
            else self.str_to_int["<|unk|>"]
            for s in preprocessed
        ]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return text

In [8]:
tokenizer = SimpleTokenizerV2(vocab)

text = "Hello, do you like tea?"
encoded = tokenizer.encode(text)

print("Encoded:", encoded)
print("Decoded:", tokenizer.decode(encoded))

Encoded: [1159, 5, 362, 1155, 642, 1000, 10]
Decoded: <|unk|> , do you like tea ?


## Step 2: Building the Vocabulary

The vocabulary built from this text contains **1,159 unique tokens**. 
After adding the two special tokens `<|unk|>` and `<|endoftext|>` 
the final vocabulary size is **1,161 tokens**.

Special tokens are critical for LLMs and agentic systems because:
- `<|unk|>` handles words the model has never seen before
- `<|endoftext|>` signals boundaries between documents during training

When the tokenizer encodes "Hello, do you like tea?" it returns 
`[1159, 5, 362, 1155, 642, 1000, 10]` and decodes back to 
`<|unk|> , do you like tea ?` — "Hello" becomes `<|unk|>` because 
it was not present in the training vocabulary.

## Why does tokenization matter for LLMs?

Tokenization is the first step in any language model pipeline. 
Before a model can process text, it needs to convert words into 
numbers. Without this step, the model has no way to perform 
mathematical operations on language...

In [9]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        # 1) Tokenizar todo el texto
        token_ids = tokenizer.encode(txt)
        
        # 2) Convertir a tensor
        self.input_ids = []
        self.target_ids = []
        
        # Sliding window: tomamos chunks de longitud max_length
        # y avanzamos 'stride' tokens cada vez
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, tokenizer, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True):
    dataset = GPTDatasetV1(txt=txt, tokenizer=tokenizer, max_length=max_length, stride=stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
    return dataloader

In [10]:
dataloader = create_dataloader_v1(
    txt=raw_text, 
    tokenizer=tokenizer, 
    batch_size=2, 
    max_length=8, 
    stride=4, 
    shuffle=False,
    drop_last=False
)

batch = next(iter(dataloader))
inputs, targets = batch

print("inputs shape:", inputs.shape)
print("targets shape:", targets.shape)
print("inputs[0]:", inputs[0])
print("targets[0]:", targets[0])

inputs shape: torch.Size([2, 8])
targets shape: torch.Size([2, 8])
inputs[0]: tensor([  55,   46,  154, 1028,   59,   39,  839,  119])
targets[0]: tensor([  46,  154, 1028,   59,   39,  839,  119,  263])


## Why do embeddings encode meaning and how do they relate to neural networks?

Embeddings are dense numerical vectors that represent tokens in 
a high-dimensional space. Tokens with similar meanings end up 
closer together in this space. Neural networks learn these 
representations during training by adjusting weights...

In [11]:
# Experiment: changing max_length and stride
from torch.utils.data import DataLoader

configs = [
    {"max_length": 4, "stride": 4},   # no overlap
    {"max_length": 4, "stride": 2},   # 50% overlap
    {"max_length": 4, "stride": 1},   # maximum overlap
    {"max_length": 8, "stride": 4},   # longer context
]

for config in configs:
    dataset = GPTDatasetV1(raw_text, tokenizer, 
                           max_length=config["max_length"], 
                           stride=config["stride"])
    print(f"max_length={config['max_length']}, stride={config['stride']} → {len(dataset)} samples")

max_length=4, stride=4 → 1162 samples
max_length=4, stride=2 → 2323 samples
max_length=4, stride=1 → 4645 samples
max_length=8, stride=4 → 1161 samples


## Step 3: Experiment — max_length and stride

| max_length | stride | samples |
|------------|--------|---------|
| 4          | 4      | 1,162   |
| 4          | 2      | 2,323   |
| 4          | 1      | 4,645   |
| 8          | 4      | 1,161   |

When `stride == max_length` there is no overlap and we get the minimum 
number of samples. When `stride = 1` (maximum overlap) we get almost 
one sample per token — 4,645 samples from just 4,649 tokens.

Overlap is useful because it ensures that context at the boundaries 
of chunks is not lost. For example with `max_length=4, stride=2`, 
each new window shares 2 tokens with the previous one, which helps 
the model learn relationships that span chunk boundaries.

## Experiment results: max_length and stride

When stride < max_length, windows overlap and we get more samples.
This overlap is useful because it ensures that important context 
at the boundaries of chunks is not lost during training...

In [12]:
import torch
import torch.nn as nn

vocab_size = len(vocab)
context_length = 8   
embedding_dim = 256 
token_embedding_layer = nn.Embedding(vocab_size, embedding_dim)
[vocab_size, embedding_dim]
pos_embedding_layer = nn.Embedding(context_length, embedding_dim)
batch = next(iter(dataloader))
inputs, targets = batch

print("Shape inputs:", inputs.shape)  # (batch_size, context_length)
token_embeddings = token_embedding_layer(inputs)

positions = torch.arange(context_length)
pos_embeddings = pos_embedding_layer(positions)

# Expandimos posiciones para batch
pos_embeddings = pos_embeddings.unsqueeze(0)

# Sumamos token + posición
input_embeddings = token_embeddings + pos_embeddings

print("Token embeddings shape:", token_embeddings.shape)
print("Pos embeddings shape:", pos_embeddings.shape)
print("Input embeddings shape:", input_embeddings.shape)

Shape inputs: torch.Size([2, 8])
Token embeddings shape: torch.Size([2, 8, 256])
Pos embeddings shape: torch.Size([1, 8, 256])
Input embeddings shape: torch.Size([2, 8, 256])


## Step 4: Token and Positional Embeddings

| Tensor               | Shape         |
|----------------------|---------------|
| inputs               | [2, 8]        |
| token embeddings     | [2, 8, 256]   |
| positional embeddings| [1, 8, 256]   |
| final input embeddings | [2, 8, 256] |

Each token ID is mapped to a vector of 256 dimensions. 
The batch of 2 sequences of 8 tokens each becomes a tensor 
of shape `[2, 8, 256]` — this is what actually enters the neural network.

**Why do embeddings encode meaning and how do they relate to neural networks?**

Embeddings are learned during training. The network adjusts the 256 values 
for each token so that tokens appearing in similar contexts end up with 
similar vectors. This is how meaning gets encoded mathematically — not by 
rules, but by exposure to millions of examples. The positional embeddings 
are added on top so the model also knows the order of tokens in the sequence, 
since transformers process all tokens in parallel and have no built-in sense of position.

## Key takeaways

- Tokenization converts raw text into integer IDs
- The sliding window creates training samples with context
- Overlap (stride < max_length) increases samples and preserves boundary context
- Embeddings transform token IDs into dense vectors the model can learn from 